# UIEDR MOD — Full DUO CUDA Training

**Before running any cell:**
1. Upload `UIEDR_MOD_CUDA_FINAL_TRAINING.zip` as a Kaggle Dataset.
2. Attach that dataset to this notebook.
3. Set **Settings → Accelerator → GPU T4 x2** (or any NVIDIA GPU).
4. Set `DATASET_SLUG` below to the slug of the dataset you created.

Run cells **in order**. Each cell has a hard stop if its precondition fails.

## STEP 1 — GPU verification

In [ ]:
import torch
import subprocess

subprocess.run(["nvidia-smi"], check=True)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is unavailable. STOP. Enable GPU in Settings → Accelerator.")

print("GPU:", torch.cuda.get_device_name(0))
print("GPU count:", torch.cuda.device_count())

## STEP 2 — Extract project

In [ ]:
from pathlib import Path
import subprocess

# ── SET THIS to your Kaggle Dataset slug ──────────────────────────────────────
DATASET_SLUG = "uiedr-mod-cuda-final"   # e.g. "akshatgupta-uiedr-mod-cuda-final"
# ─────────────────────────────────────────────────────────────────────────────

PACKAGE = f"/kaggle/input/{DATASET_SLUG}/UIEDR_MOD_CUDA_FINAL_TRAINING.zip"
WORKDIR = Path("/kaggle/working/uiedr")

if not Path(PACKAGE).exists():
    raise FileNotFoundError(
        f"Package not found: {PACKAGE}\n"
        "Check that DATASET_SLUG matches the slug of the attached Kaggle Dataset."
    )

WORKDIR.mkdir(parents=True, exist_ok=True)
subprocess.run(["unzip", "-q", PACKAGE, "-d", str(WORKDIR)], check=True)
%cd /kaggle/working/uiedr
print("Working directory:", Path.cwd())
print("Files present:", [p.name for p in Path(".").iterdir()])

## STEP 3 — Verify dataset counts and classes

In [ ]:
import json
from pathlib import Path

EXPECTED_CLASSES = ["holothurian", "echinus", "scallop", "starfish"]

duo = Path("dataset/DUO")
train_images = list((duo / "images/train").glob("*"))
test_images  = list((duo / "images/test").glob("*"))
train_ann = json.loads((duo / "annotations/instances_train.json").read_text())
test_ann  = json.loads((duo / "annotations/instances_test.json").read_text())

print("train images     :", len(train_images))
print("train annotations:", len(train_ann["annotations"]))
print("test  images     :", len(test_images))
print("test  annotations:", len(test_ann["annotations"]))

assert len(train_images)              == 6671,  f"Expected 6671 train images, got {len(train_images)}"
assert len(train_ann["annotations"]) == 63998, f"Expected 63998 train anns, got {len(train_ann['annotations'])}"
assert len(test_images)               == 1111,  f"Expected 1111 test images, got {len(test_images)}"
assert len(test_ann["annotations"])  == 10517, f"Expected 10517 test anns, got {len(test_ann['annotations'])}"

import yaml
cfg = yaml.safe_load(open("config.yaml"))
assert cfg["classes"] == EXPECTED_CLASSES, f"Class mismatch: {cfg['classes']}"
print("Classes:", cfg["classes"])
print("\nAll dataset checks PASSED. Proceed to training.")

## STEP 4 — Verify dependencies

In [ ]:
import cv2, numpy, torchvision
print("NumPy:", numpy.__version__)
print("OpenCV:", cv2.__version__)
print("TorchVision:", torchvision.__version__)
print("All dependencies OK.")

## STEP 5 — Full CUDA training

`kaggle_train.py` handles:
- CUDA guard (stops if no GPU)
- dataset count guard (stops if counts wrong)
- checkpoint guard (stops if old `.pt` files exist in `checkpoints/`)
- OOM recovery (halves batch size, resumes from `latest.pt`, keeps all 6671 images)
- 50-epoch target
- full 1111-image evaluation
- real visual inference on 994.jpg, 995.jpg + 8 more
- checkpoint provenance validation

**Do not run this cell if any check above failed.**

In [ ]:
!python kaggle_train.py

## STEP 6 — Verify outputs

In [ ]:
import json
from pathlib import Path

# ── Checkpoint ────────────────────────────────────────────────────────────────
best = Path("checkpoints/best.pt")
assert best.exists(), "FAILED: checkpoints/best.pt not found"
print(f"best.pt size: {best.stat().st_size / 1e6:.1f} MB")

import torch
ck = torch.load(str(best), map_location="cpu", weights_only=False)
meta = ck.get("training_metadata", {})
print("epoch            :", ck.get("epoch"))
print("device           :", meta.get("device"))
print("gpu_name         :", meta.get("gpu_name"))
print("num_train_samples:", meta.get("num_training_samples"))
print("created_utc      :", meta.get("created_utc"))
assert meta.get("device") == "cuda",  "FAILED: checkpoint was not trained on CUDA"
assert meta.get("num_training_samples") == 6671, "FAILED: not trained on all 6671 images"

# ── Metrics ───────────────────────────────────────────────────────────────────
metrics_json = Path("reports/final_metrics.json")
assert metrics_json.exists(), "FAILED: reports/final_metrics.json not found"
report = json.loads(metrics_json.read_text())
print("\n── Final metrics ──")
for key in ("precision", "recall", "f1", "mAP@0.50", "mAP@0.50:0.95"):
    print(f"  {key:<20}: {report.get(key, 'n/a'):.4f}")
print("  images_evaluated :", report["dataset"]["images_evaluated"])
print("  images_with_dets :", report["dataset"]["images_with_detections"])
print("  per-class counts :", report["dataset"]["predicted_class_counts"])
assert report["dataset"]["images_evaluated"] == 1111, "FAILED: not all 1111 test images evaluated"

# ── Visualizations ────────────────────────────────────────────────────────────
vis_dir = Path("outputs/visualizations/final_test")
vis_files = list(vis_dir.glob("*.jpg")) + list(vis_dir.glob("*.png"))
print(f"\nVisualization files: {len(vis_files)}")
assert len(vis_files) >= 10, f"FAILED: expected >=10 visualizations, got {len(vis_files)}"
names = {f.name for f in vis_files}
assert any("994" in n for n in names), "FAILED: 994.jpg visualization missing"
assert any("995" in n for n in names), "FAILED: 995.jpg visualization missing"

print("\nAll output checks PASSED.")

## STEP 7 — Print final report

Copy this output into the final report.

In [ ]:
import json, torch
from pathlib import Path

ck   = torch.load("checkpoints/best.pt", map_location="cpu", weights_only=False)
meta = ck.get("training_metadata", {})
rep  = json.loads(Path("reports/final_metrics.json").read_text())
log  = [json.loads(l) for l in Path("outputs/metrics/train_log_kaggle.jsonl").read_text().splitlines() if l.strip()]
last = log[-1]
best_epoch = min(log, key=lambda r: r.get("val_total", float("inf")))["epoch"]

vis_index = json.loads(Path("outputs/visualizations/final_test/inference_results.json").read_text())
det_994 = next((x for x in vis_index if x["image"] == "994.jpg"), None)
det_995 = next((x for x in vis_index if x["image"] == "995.jpg"), None)

print("=" * 60)
print("UIEDR MOD — FINAL CUDA TRAINING REPORT")
print("=" * 60)
print(f"CUDA GPU name          : {meta.get('gpu_name')}")
print(f"CUDA available         : True")
print(f"Training images        : {meta.get('num_training_samples')}")
print(f"Training annotations   : 63998")
print(f"Test images            : {rep['dataset']['images_evaluated']}")
print(f"Test annotations       : 10517")
print(f"Epochs completed       : {last['epoch']}")
print(f"Best epoch             : {best_epoch}")
print(f"Final train loss       : {last.get('train_total', 'n/a')}")
print(f"Final val loss         : {last.get('val_total', 'n/a')}")
print(f"RPN objectness loss    : {last.get('train_rpn_objectness', 'n/a')}")
print(f"RPN bbox loss          : {last.get('train_rpn_box', 'n/a')}")
print(f"ROI cls loss           : {last.get('train_roi_cls', 'n/a')}")
print(f"ROI bbox loss          : {last.get('train_roi_box', 'n/a')}")
print(f"Precision              : {rep['precision']:.4f}")
print(f"Recall                 : {rep['recall']:.4f}")
print(f"F1                     : {rep['f1']:.4f}")
print(f"mAP@0.50               : {rep['mAP@0.50']:.4f}")
print(f"mAP@0.50:0.95          : {rep['mAP@0.50:0.95']:.4f}")
print(f"Images with predictions: {rep['dataset']['images_with_detections']}")
print(f"Per-class counts       : {rep['dataset']['predicted_class_counts']}")
print(f"994.jpg detections     : {det_994['num_detections'] if det_994 else 'NOT TESTED'}")
print(f"995.jpg detections     : {det_995['num_detections'] if det_995 else 'NOT TESTED'}")
print(f"New best.pt path       : checkpoints/best.pt")
print(f"Final metrics path     : reports/final_metrics.json")
print(f"Visualizations path    : outputs/visualizations/final_test/")
print("=" * 60)
print("NOTE: The DUO official test split is used for validation/checkpoint")
print("selection AND final evaluation. These metrics are NOT from an")
print("untouched independent holdout.")
print("=" * 60)

## STEP 8 — Download artifacts

Use the Kaggle file browser (Output tab) to download:

- `/kaggle/working/uiedr/checkpoints/best.pt`
- `/kaggle/working/uiedr/reports/final_metrics.json`
- `/kaggle/working/uiedr/reports/final_metrics.txt`
- `/kaggle/working/uiedr/outputs/visualizations/final_test/` (folder)

Then follow the local Streamlit verification steps.